# Video tutorial: dynamic camera

This notebook uses only data embedded in one 4DGS video. It selects a dynamic camera with a fixed random seed. It renders RGB, depth, optical flow, and hand point clouds for the first 30 frames. It writes four MP4 files and displays them inline. It also draws an instance overlay at time 0.

## Dependencies

The companion `requirements.txt` lists every direct dependency of this tutorial:

| Package | Purpose |
| --- | --- |
| `graciasdk` | Load and render the video on the GPU |
| `numpy` | Process RGB, depth, and coverage arrays |
| `av` | Write H.264 MP4 files |
| `matplotlib` | Draw and save the instance overlay |
| `jupyterlab`, `ipykernel`, `ipython` | Run the notebook and display videos inline |

From the repository root, start an isolated environment with one command:

```bash
uv run --isolated --no-project --python 3.12 --find-links wheels --with-requirements examples/notebooks/requirements.txt jupyter lab examples/notebooks/video_dynamic_camera.ipynb
```

`uv` selects the local SDK wheel for the host platform. The notebook contains no installation cell and does not change its active kernel.

In [ ]:
from importlib.metadata import version
from fractions import Fraction
from pathlib import Path
import random

import av
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, Video, display
from matplotlib.patches import Patch

from graciasdk import HAND_EDGES, GraciaSDK, draw_points, flow_colors

{
    package: version(package)
    for package in ("graciasdk", "numpy", "av", "matplotlib")
}

## Settings

Set `VIDEO_PATH` to a video with a dynamic camera, stable splat IDs for flow, and `hand_*` point clouds. Change `SEED` to select another dynamic camera. `FPS` sets the time step. `LONG_SIDE` sets the render size; the camera aspect ratio stays the same, apart from the even-pixel round-off for H.264. Flow requires one extra source frame after the last output frame.

In [ ]:
VIDEO_PATH = Path("/data/local_Shot13_Take2_42_1565_1789571697/Shot13_Take2_42_682_grvpy.mint")
OUTPUT_DIR = VIDEO_PATH.parent / "dynamic_camera_preview"

FRAME_COUNT = 30
FPS = 30.0
LONG_SIDE = 800
SEED = 2026
MASK_THRESHOLD = 1.0 / 255.0

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(VIDEO_PATH)
if FRAME_COUNT < 1 or FPS <= 0 or LONG_SIDE < 2:
    raise ValueError("Use FRAME_COUNT >= 1, FPS > 0, and LONG_SIDE >= 2")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Render and video helpers

The depth buffer stores negative view-space Z. All frames use one depth range. Coverage makes low-opacity edges faint. The view uses the camera pose at `view.time` and waits for the scene at that time.

In [ ]:
def rgb_at(view, timestamp):
    view.time = timestamp
    rgb = view.color8[..., :3].copy()
    if view.buffering:
        raise TimeoutError(f"Frame at {timestamp:.3f}s stayed buffered")
    return rgb


def make_depth_frames(depths, coverages):
    valid_masks = [
        (coverage > 0.0) & np.isfinite(depth) & (depth != 0.0)
        for depth, coverage in zip(depths, coverages)
    ]
    if not any(mask.any() for mask in valid_masks):
        raise ValueError("The rendered depth is empty")

    far = min(float(depth[mask].min()) for depth, mask in zip(depths, valid_masks) if mask.any())
    near = max(float(depth[mask].max()) for depth, mask in zip(depths, valid_masks) if mask.any())
    scale = max(near - far, np.finfo(np.float32).eps)
    frames = []

    for depth, coverage, valid in zip(depths, coverages, valid_masks):
        gray = np.zeros_like(depth, dtype=np.float32)
        gray[valid] = (depth[valid] - far) / scale
        gray *= coverage
        gray_u8 = np.rint(np.clip(gray, 0.0, 1.0) * 255.0).astype(np.uint8)
        frames.append(np.repeat(gray_u8[..., None], 3, axis=2))

    return frames, (far, near)


def write_mp4(path, frames, fps):
    if not frames:
        raise ValueError("No frames to write")

    height, width, channels = frames[0].shape
    if channels != 3:
        raise ValueError("Expected RGB frames")

    rate = Fraction(str(fps)).limit_denominator(100_000)
    time_base = Fraction(rate.denominator, rate.numerator)

    with av.open(str(path), "w", options={"movflags": "+faststart"}) as container:
        stream = container.add_stream("libx264", rate=rate)
        stream.width = width
        stream.height = height
        stream.pix_fmt = "yuv420p"
        stream.options = {"crf": "18", "preset": "medium"}

        for index, image in enumerate(frames):
            frame = av.VideoFrame.from_ndarray(np.ascontiguousarray(image), format="rgb24")
            frame.pts = index
            frame.time_base = time_base
            for packet in stream.encode(frame):
                container.mux(packet)

        for packet in stream.encode():
            container.mux(packet)

## Load the video and select a dynamic camera

Pass the embedded `SceneCamera` directly to `sdk.view`. The view updates the camera pose with time. Hand colors stay fixed across the video.

In [ ]:
sdk = GraciaSDK()
scene = sdk.load(str(VIDEO_PATH))
scene.wait_ready(timeout=120.0)

dynamic_cameras = [camera for camera in scene.cameras if camera.is_dynamic]
if not dynamic_cameras:
    raise ValueError("The video has no dynamic camera")
if FRAME_COUNT / FPS >= scene.duration():
    raise ValueError("The video needs one extra source frame for the last flow frame")

camera_track = random.Random(SEED).choice(dynamic_cameras)
size_scale = LONG_SIDE / max(camera_track.width, camera_track.height)
WIDTH, HEIGHT = [max(2, 2 * round(size * size_scale / 2)) for size in (camera_track.width, camera_track.height)]
view = sdk.view(scene, camera_track, WIDTH, HEIGHT, wait=30.0)
hand_colors = {
    cloud.name: plt.colormaps["tab10"](index % 10)[:3]
    for index, cloud in enumerate(scene.point_clouds)
    if cloud.name.startswith("hand_")
}
if not hand_colors:
    raise ValueError("The video has no hand point clouds")

print(f"Selected camera: {camera_track.name} ({camera_track.poses_count} poses)")
print(f"Render size: {WIDTH} x {HEIGHT}")
print(f"Hands: {list(hand_colors)}")

## Render RGB, depth, flow, and hands

`view.flow(t + 1 / FPS)` gives flow from the current frame to the next frame. The SDK uses the camera pose at each time, so the flow includes both object motion and camera motion. Only splats present at both times contribute.

`flow_colors` maps direction to hue and uses the 95th percentile of flow magnitude for the scale of each frame. This makes small motion visible; color strength is not a fixed speed scale across the video. Pixels without flow coverage are black.

`view.points` and `view.camera_setup` use the same time as RGB. `draw_points` projects each hand's 21 joints and the `HAND_EDGES` bones onto RGB. It skips NaN points. The overlay has no depth test, so joints can appear in front of objects that hide them.

In [ ]:
scene.set_uint("instance_filter", 0)
rgb_frames, flow_frames, hand_frames = [], [], []
depth_values, coverage_values, flow_scales = [], [], []

for frame_index in range(FRAME_COUNT):
    timestamp = frame_index / FPS
    rgb = rgb_at(view, timestamp)
    rgb_frames.append(rgb)
    depth_values.append(view.depth.copy())
    coverage_values.append(view.coverage.copy())

    hands = rgb.copy()
    for cloud, points in view.points:
        if cloud.name in hand_colors:
            draw_points(hands, view.camera_setup, points, HAND_EDGES, hand_colors[cloud.name],
                        square=max(2, WIDTH // 250), line=max(1, WIDTH // 700))
    hand_frames.append(hands)

    flow = view.flow((frame_index + 1) / FPS)
    if flow is None:
        raise ValueError("The video has no stable splat IDs for flow")
    if flow.buffering:
        raise TimeoutError(f"Flow at {timestamp:.3f}s stayed buffered")
    colors, scale = flow_colors(flow.flow, flow.coverage)
    flow_frames.append(colors)
    flow_scales.append(scale)

    if frame_index == 0 or (frame_index + 1) % 5 == 0:
        print(f"Rendered {frame_index + 1}/{FRAME_COUNT}")

## Write and view the four videos

All videos use the same camera and output times. Each flow frame points to the next source frame. The cell embeds the videos so they play directly in Jupyter.

In [ ]:
depth_frames, (depth_far, depth_near) = make_depth_frames(depth_values, coverage_values)
print(f"Shared depth range: far={depth_far:.4f}, near={depth_near:.4f}")
print(f"Flow scale: {min(flow_scales):.3f} to {max(flow_scales):.3f} pixels per frame")

for name, frames in (("rgb", rgb_frames), ("depth", depth_frames), ("flow", flow_frames), ("hands", hand_frames)):
    assert len(frames) == FRAME_COUNT
    assert all(frame.shape == (HEIGHT, WIDTH, 3) and frame.dtype == np.uint8 for frame in frames)
    path = OUTPUT_DIR / f"camera_{camera_track.name}_first_{FRAME_COUNT}_{name}.mp4"
    write_mp4(path, frames, FPS)
    print(path)
    display(Markdown(f"### {name.upper()}"))
    display(Video(filename=str(path), embed=True, width=WIDTH, html_attributes="controls loop muted playsinline"))

## Render every instance label at time 0

The SDK renders the selected instance in white and every other splat in black. All splats still take part in visibility, so foreground objects occlude the selected instance. The mean RGB highlight is the soft instance mask. `view.coverage` describes the full scene and is not an instance mask. The code keeps the strongest highlight at each pixel and uses its strength for smooth overlay edges.

In [ ]:
scene.set_uint("instance_filter", 0)
first_rgb = rgb_at(view, 0.0)
instance_names = scene.instance_names()
instance_ids = sorted(instance_names)

label_map = np.zeros((HEIGHT, WIDTH), dtype=np.int32)
best_highlight = np.zeros((HEIGHT, WIDTH), dtype=np.float32)

for index, instance_id in enumerate(instance_ids, start=1):
    scene.set_uint("instance_filter", instance_id)
    highlight_rgb = view.color[..., :3].astype(np.float32)
    highlight = highlight_rgb.mean(axis=-1)
    take = (highlight >= MASK_THRESHOLD) & (highlight > best_highlight)
    label_map[take] = instance_id
    best_highlight[take] = highlight[take]
    if index % 10 == 0 or index == len(instance_ids):
        print(f"Rendered labels {index}/{len(instance_ids)}")

scene.set_uint("instance_filter", 0)
visible_ids = [instance_id for instance_id in instance_ids if np.any(label_map == instance_id)]
positions = np.linspace(0.05, 0.95, len(instance_ids))
colors = {
    instance_id: np.asarray(plt.colormaps["turbo"](position)[:3])
    for instance_id, position in zip(instance_ids, positions)
}

overlay = first_rgb.astype(np.float32) / 255.0
for instance_id in visible_ids:
    mask = label_map == instance_id
    alpha = (0.55 * best_highlight[mask])[:, None]
    overlay[mask] = (1.0 - alpha) * overlay[mask] + alpha * colors[instance_id]

fig, (rgb_axis, overlay_axis) = plt.subplots(1, 2, figsize=(18, 8))
rgb_axis.imshow(first_rgb)
rgb_axis.set_title(f"RGB · camera {camera_track.name} · time 0")
rgb_axis.axis("off")

overlay_axis.imshow(np.clip(overlay, 0.0, 1.0))
overlay_axis.set_title(f"Instance overlay · {len(visible_ids)}/{len(instance_ids)} visible")
overlay_axis.axis("off")

handles = [
    Patch(color=colors[instance_id], label=f"{instance_id}: {instance_names[instance_id]}")
    for instance_id in visible_ids
]
if handles:
    overlay_axis.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)

overlay_image = OUTPUT_DIR / f"camera_{camera_track.name}_time_0_instances.png"
fig.tight_layout()
fig.savefig(overlay_image, dpi=150, bbox_inches="tight")
plt.show()

print(f"Rendered all {len(instance_ids)} labels")
print("Visible labels:", [f"{instance_id}:{instance_names[instance_id]}" for instance_id in visible_ids])
print(f"Overlay: {overlay_image}")